# Submission from saved ConvNeXt weights — no training

Run this notebook in Kaggle to restore `best.pt` and predict the test set after a kernel restart.
It does **not** need train/validation loaders, folds, training history or W&B credentials.

1. Attach the Cats and Dogs competition data.
2. Keep `best.pt` in the current Kaggle session, or attach a saved notebook Output/dataset containing it.
   If several checkpoints are available, set `CHECKPOINT_PATH` explicitly below.
3. Run all cells here. Download `submission.csv` from Output and submit it on Kaggle.

The saved `.ipynb` and displayed metrics do not contain model weights. If `best.pt` was lost,
this notebook stops with instructions; it never starts training automatically.

In [1]:
# Run only if packages are missing, then restart the kernel before continuing.
# %pip install -q timm

## 1. Imports and paths

Use zero DataLoader worker processes and plain tqdm to avoid the previous notebook
multiprocessing cleanup errors. The default checkpoint path matches the full training run.

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, FileLink
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
import timm

CHECKPOINT_PATH = Path('/kaggle/working/convnext_tiny_fb_in1k_fold0_full/best.pt')
DATA_DIR = Path('/kaggle/input/competitions/cats-and-dogs')
OUTPUT_DIR = Path('/kaggle/working')
BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cuda


## 2. Restore the saved model

Prefer the specified full-run checkpoint. If it is absent, search saved Kaggle outputs.
Rebuild the network with `pretrained=False` because the complete trained state is loaded
from `best.pt`; this avoids downloading the initial ImageNet weights again. Training in
`cnn_2.ipynb` still starts with `convnext_tiny.fb_in1k` and `pretrained=True`.

In [3]:
if not CHECKPOINT_PATH.is_file():
    candidates = set()
    for search_root in [Path('/kaggle/working'), Path('/kaggle/input')]:
        for directory, subdirs, files in os.walk(search_root):
            subdirs[:] = [name for name in subdirs if name not in {'train', 'test', 'images', 'wandb'}]
            if 'best.pt' in files:
                candidates.add(Path(directory) / 'best.pt')
    candidates = sorted(candidates)
    if len(candidates) == 1:
        CHECKPOINT_PATH = candidates[0]
    elif len(candidates) > 1:
        raise ValueError(f'Set CHECKPOINT_PATH to the desired full-run weights: {candidates}')
    else:
        raise FileNotFoundError(
            'best.pt was not found in this Kaggle session or attached inputs. '
            'Attach the saved training notebook Output containing best.pt, or upload your downloaded best.pt. '
            'The notebook file and W&B metrics do not restore weights. No training has been started.'
        )

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
config = checkpoint['config']
assert config['model_name'] == 'convnext_tiny.fb_in1k', 'Use the ImageNet-1k ConvNeXt checkpoint.'
LABEL_COLS = list(config['label_cols'])
assert len(LABEL_COLS) > 0 and len(set(LABEL_COLS)) == len(LABEL_COLS)
THRESHOLD = float(config.get('threshold', 0.5))
assert 0 < THRESHOLD < 1
model = timm.create_model(config['model_name'], pretrained=False, num_classes=len(LABEL_COLS))
model.load_state_dict(checkpoint['model'], strict=True)
model = model.to(DEVICE).eval()
data_config = dict(config['data_config'])
eval_transform = timm.data.create_transform(**data_config, is_training=False)
print('Restored:', CHECKPOINT_PATH)
print('Best epoch:', checkpoint['epoch'], '| saved validation metrics:', checkpoint['metrics'])

Restored: /kaggle/working/convnext_tiny_fb_in1k_fold0_full/best.pt
Best epoch: 5 | saved validation metrics: {'macro_f1': 0.9701310012664232, 'macro_roc_auc': 0.999909977643553, 'val_loss': 0.00851320562670403, 'val_accuracy': 0.9910130718954249}


## 3. Test data and loader

Read test.csv and reuse the exact evaluation transform recorded in the checkpoint.
No split creation, label changes or validation tuning is performed.

In [4]:
if not (DATA_DIR / 'test.csv').is_file():
    data_candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [name for name in subdirs if name not in {'train', 'test', 'images', 'wandb'}]
        root = Path(directory)
        if 'test.csv' in files and 'sample_submission.csv' in files and (root / 'test').is_dir():
            sample_columns = pd.read_csv(root / 'sample_submission.csv', nrows=0).columns
            if set(sample_columns) == {'image_id', *LABEL_COLS}:
                data_candidates.append(root)
    if len(data_candidates) != 1:
        raise FileNotFoundError(f'Attach competition data or set DATA_DIR explicitly. Matching directories: {data_candidates}')
    DATA_DIR = data_candidates[0]
test = pd.read_csv(DATA_DIR / 'test.csv', dtype={'image_id': str})
assert len(test) > 0 and test.image_id.notna().all() and test.image_id.is_unique

class TestImages(Dataset):
    def __len__(self):
        return len(test)

    def __getitem__(self, index):
        image_path = DATA_DIR / 'test' / f'{test.iloc[index].image_id}.jpg'
        with Image.open(image_path) as image:
            return eval_transform(image.convert('RGB'))

test_loader = DataLoader(TestImages(), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
print('Test images:', len(test), '| data:', DATA_DIR)

Test images: 18673 | data: /kaggle/input/competitions/cats-and-dogs


## 4. Predict, validate and export

Apply sigmoid, then the checkpoint's fixed threshold. Match sample-submission row order
and save integer 0/1 predictions. The displayed checkpoint metrics are validation results;
the test score is available only after submitting to Kaggle.

In [5]:
probabilities = []
with torch.inference_mode():
    for images in tqdm(test_loader, desc='test inference'):
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(images.to(DEVICE))
        probabilities.append(logits.float().sigmoid().cpu().numpy())
probabilities = np.concatenate(probabilities)
assert probabilities.shape == (len(test), len(LABEL_COLS))
assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
probability_frame = pd.DataFrame(probabilities, columns=LABEL_COLS)
probability_frame.insert(0, 'image_id', test.image_id.to_numpy())
submission = probability_frame.copy()
submission[LABEL_COLS] = (probabilities >= THRESHOLD).astype(np.int32)

sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'image_id': str})
expected_columns = ['image_id', 'Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
assert set(sample.columns) == set(expected_columns) == set(submission.columns)
assert sample.image_id.notna().all() and sample.image_id.is_unique
assert set(sample.image_id) == set(test.image_id)
submission = sample[['image_id']].merge(submission, on='image_id', how='left', validate='one_to_one')
submission = submission[expected_columns]
assert len(submission) == len(test) and not submission.isna().any().any()
assert submission[LABEL_COLS].isin([0, 1]).all().all()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
probability_frame.to_csv(OUTPUT_DIR / 'test_probabilities.csv', index=False)
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print('Saved:', OUTPUT_DIR / 'submission.csv', '| rows:', len(submission))
display(FileLink(str(OUTPUT_DIR / 'submission.csv')))
display(submission.head())

test inference: 100%|██████████| 584/584 [04:59<00:00,  1.95it/s]


Saved: /kaggle/working/submission.csv | rows: 18673


/kaggle/working/submission.csv

,image_id,Cat,Dog,Moris,Motya,Biatrix
0,12c79b0a4bd444ca7e5f730b86824b9279d3cd24,0,0,0,0,0
1,dd9440332972c9c5cecb5bf3cfb3329d6b2e422c,0,1,0,0,0
2,ac89f642faef21c92d622d882943e9065600354a,0,0,0,0,0
3,7245e7046ae54da95dbddc24ae814b3c61cc6237,1,0,0,0,0
4,d2b76179615de57b54cb00d094fba50810d96127,1,0,0,0,0
